In [ ]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:

# 0 Initialize the SAME embedding model used in the first notebook
# This is required so Chroma can understand the mathematical vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

# Point to your storage folder
persist_directory = "../chroma_db"

# Define the 'vectorstore' variable by loading the folder
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

print(f"Success: vectorstore is now defined. Found {len(vectorstore.get()['ids'])} chunks.")

In [ ]:
# 1. Initialize the Local Docker Model
# Docker Model Runner serves an OpenAI-compatible API on port 12434
llm = ChatOpenAI(
    model="ai/qwen3:0.6B-F16",
    base_url="http://localhost:12434/engines/v1",
    api_key="not-needed", 
    temperature=0
)

In [ ]:
# 2. Setup the Chain (Use your existing prompt logic)
system_prompt = (
    "You are a SQL expert. Use the following context to answer the question. "
    "Include code examples. If the answer isn't in the context, say so."
    "\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
# 3. Create the Chain (Using the same vectorstore you already built)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("Local Qwen3 model is now powering your SQL Bot!")